# Chapter 5: Sector Proxy vs Representative Stocks

*If you want exposure to a sector, is one representative stock enough, or do you need a small basket?*

For each sector this chapter:

1. builds the **sector proxy**: the dollar-volume weighted return of all the sector's stocks;
2. keeps the **liquid** stocks and ranks them by how closely they follow the proxy over time;
3. compares the **best single stock** and the **Top-5 basket** with the proxy (growth index and tracking error);
4. draws a **UMAP map** of the sector's stocks with **HDBSCAN** clusters, to see whether the sector moves as one block or splits into groups.

- **Inputs:** `data_processed/stocks_sect.parquet`, and `metadata_clean.parquet` for the company names.
- **Output:** the 33 interactive Chapter 5 figures of the website (UMAP map, index comparison and tracking error for each of the 11 sectors) in `website/figures/chapter5/`.

**Contents**
1. Setup
2. Method
3. Run all sectors
4. One sector in the notebook
5. Numbers quoted in the website text

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from src.chapters.chapter5 import (
    company_key,
    comparison_stats,
    correlation_distance,
    equal_weight_returns,
    hdbscan_clusters,
    liquid_stocks,
    plot_index,
    plot_tracking_error,
    plot_umap,
    representativeness,
    rolling_tracking_error,
    sector_panel,
    select_representatives,
    stock_correlations,
    umap_embedding,
)
from src.utils.figures import save_html_figure

PROCESSED_DIR = Path("data_processed")
FIG_SECTION = "chapter5"

START_DATE = "2005-01-01"     # the last 15 years of data: a representative must be investable over a long, recent period
END_DATE = "2020-04-01"
REP_WINDOW = 90               # rolling window (days) for the correlation with the proxy
LIFETIME_ALPHA = 2            # penalty for stocks that exist for only part of the period
TOP_N = 5                     # size of the basket
TE_WINDOWS = (60, 120)        # rolling tracking-error windows (days)
MIN_OVERLAP = 252             # a stock-stock correlation needs at least one year of common days
MAX_MISSING_CORR = 0.2        # stocks missing more than 20% of their correlations are left off the map
UMAP_NEIGHBORS, UMAP_MIN_DIST = 10, 0.01
MIN_CLUSTER_SIZE = 5
SHOW_SECTOR = "Technology"    # sector whose figures are displayed in this notebook


def slug(sector):
    return sector.lower().replace(" ", "_")

## 2. Method

**Sector proxy.** The dollar-volume weighted daily return of all the sector's stocks (`Weight × Return` from `stocks_sect`), the same sector series as in the previous chapters.

**Liquid stocks.** A stock is kept if it trades on at least 80% of its days and its average dollar volume is above the sector's 20% quantile.

**Ranking.** For each stock, the average of its rolling 90-day correlation with the proxy (`rep_score`), multiplied by `lifetime²`, where `lifetime` is the share of the period with a return. A stock that only exists for a few years cannot stand in for the sector over 15 years. Ties go to the lower volatility.

**Single stock and basket.** The best-ranked stock, and an equal-weighted basket of the Top 5 **companies**: a second share class of a company already in the basket (e.g. GOOGL after GOOG) is skipped, since it would add the same company twice.

**Tracking error.** The standard deviation of the daily difference with the proxy, annualised (× √252), over the whole period and on 60- and 120-day rolling windows.

**Structure map.** Correlations between all pairs of liquid stocks (at least one year of common days) → angular distance √(2(1 − ρ)) → 2-D UMAP map → HDBSCAN clusters on the map. Stocks missing too many correlations (too little common history) are left off the map.

Caveat: the representative and the basket are chosen and evaluated on the same period, so their fit is an in-sample best case.

## 3. Run all sectors

For each sector, the three website figures are saved as interactive HTML (dark theme), and the numbers of the write-ups are collected.

In [2]:
stocks_sect = pd.read_parquet(
    PROCESSED_DIR / "stocks_sect.parquet",
    columns=["Date", "Symbol", "Sector", "Return", "Weight", "DollarVolume"],
)
sectors = sorted(stocks_sect["Sector"].unique())

metadata = pd.read_parquet(PROCESSED_DIR / "metadata_clean.parquet", columns=["symbol", "security_name"])
company = metadata.set_index("symbol")["security_name"].map(company_key)

rows, figures = [], {}
for sector in sectors:
    returns, dollar_volume, proxy = sector_panel(stocks_sect, sector, START_DATE, END_DATE)
    liquid = liquid_stocks(dollar_volume)
    returns, dollar_volume = returns[liquid], dollar_volume[liquid]

    # Ranking, single stock and basket
    ranking = select_representatives(returns, representativeness(returns, proxy, REP_WINDOW, REP_WINDOW * 2 // 3), TOP_N, LIFETIME_ALPHA, company)
    basket_tickers = ranking.index[ranking["selected"]].tolist()
    single_ticker = basket_tickers[0]
    single, basket = returns[single_ticker], equal_weight_returns(returns, basket_tickers)

    # Tracking error
    te = {}
    for w in TE_WINDOWS:
        te[f"{single_ticker} ({w}d)"] = rolling_tracking_error(single, proxy, w, w * 2 // 3)
        te[f"Basket ({w}d)"] = rolling_tracking_error(basket, proxy, w, w * 2 // 3)
    single_stats = comparison_stats(single, proxy, te[f"{single_ticker} ({TE_WINDOWS[0]}d)"])
    basket_stats = comparison_stats(basket, proxy, te[f"Basket ({TE_WINDOWS[0]}d)"])

    # Structure map
    corr = stock_correlations(returns, MIN_OVERLAP)
    missing = corr.isna().sum() / max(len(corr) - 1, 1)
    mapped = corr.index[missing <= MAX_MISSING_CORR]
    embedding = umap_embedding(correlation_distance(corr.loc[mapped, mapped]), UMAP_NEIGHBORS, UMAP_MIN_DIST)
    clusters = hdbscan_clusters(embedding, MIN_CLUSTER_SIZE)
    sizes = clusters[clusters != "Outlier"].value_counts()

    figures[sector] = {
        "umap": plot_umap(embedding, clusters, ranking, dollar_volume.mean(), sector, theme="dark"),
        "index": plot_index(proxy, single, basket, single_ticker, basket_tickers, sector, theme="dark"),
        "te": plot_tracking_error(te, sector, theme="dark"),
    }
    for kind, fig in figures[sector].items():
        save_html_figure(fig, FIG_SECTION, f"{slug(sector)}_{kind}")

    rows.append({
        "sector": sector, "liquid stocks": len(liquid), "stocks on the map": len(mapped),
        "clusters": len(sizes), "largest cluster share": round(sizes.iloc[0] / len(mapped), 3) if len(sizes) else None,
        "single": single_ticker, "basket": ", ".join(basket_tickers),
        **{f"single {k}": round(v, 3) for k, v in single_stats.items()},
        **{f"basket {k}": round(v, 3) for k, v in basket_stats.items()},
    })
    print(f"{sector}: single {single_ticker}, basket {basket_tickers}")

print(f"Saved {3 * len(sectors)} figures to website/figures/{FIG_SECTION}/")

Basic Materials: single FCX, basket ['FCX', 'BHP', 'VALE', 'RIO', 'SCCO']
Communication Services: single GOOG, basket ['GOOG', 'NFLX', 'BIDU', 'DIS', 'CMCSA']
Consumer Cyclical: single AMZN, basket ['AMZN', 'MAR', 'HD', 'LVS', 'MGM']
Consumer Defensive: single WMT, basket ['WMT', 'PG', 'TGT', 'COST', 'CL']
Energy: single COP, basket ['COP', 'CVX', 'EOG', 'OXY', 'HP']
Financial Services: single JPM, basket ['JPM', 'C', 'GS', 'WFC', 'USB']
Healthcare: single AMGN, basket ['AMGN', 'PFE', 'BIIB', 'GILD', 'LLY']
Industrials: single HON, basket ['HON', 'ITW', 'EMR', 'ETN', 'DOV']
Real Estate: single SPG, basket ['SPG', 'VNO', 'FRT', 'REG', 'PLD']
Technology: single AAPL, basket ['AAPL', 'INTC', 'CSCO', 'MSFT', 'ADI']
Utilities: single AEP, basket ['AEP', 'XEL', 'DTE', 'AEE', 'PNW']
Saved 33 figures to website/figures/chapter5/


## 4. One sector in the notebook

The website figures for `SHOW_SECTOR` (the other sectors are in `website/figures/chapter5/`).

In [3]:
for fig in figures[SHOW_SECTOR].values():
    fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", font_color="#0b0b0b")
    fig.show()

## 5. Numbers quoted in the website text

Each sector's write-up in `website/index.html` quotes these values: number of clusters and share of the largest one, the single stock and the basket, their correlation with the proxy, annualised tracking error, and the 95th percentile / maximum of the 60-day rolling tracking error. After a new run, update the text with them.

In [4]:
pd.DataFrame(rows).set_index("sector")

,liquid stocks,stocks on the map,clusters,largest cluster share,single,basket,single corr,single te_ann,single te_p95,single te_max,basket corr,basket te_ann,basket te_p95,basket te_max
sector,,,,,,,,,,,,,,
Basic Materials,114,114,2,0.781,FCX,"FCX, BHP, VALE, RIO, SCCO",0.846,0.312,0.601,0.933,0.922,0.167,0.259,0.366
Communication Services,93,93,2,0.860,GOOG,"GOOG, NFLX, BIDU, DIS, CMCSA",0.833,0.169,0.246,0.316,0.851,0.151,0.247,0.336
Consumer Cyclical,230,230,2,0.943,AMZN,"AMZN, MAR, HD, LVS, MGM",0.672,0.288,0.454,0.550,0.824,0.188,0.349,0.562
Consumer Defensive,99,99,7,0.162,WMT,"WMT, PG, TGT, COST, CL",0.663,0.159,0.229,0.285,0.839,0.101,0.171,0.238
Energy,107,107,2,0.944,COP,"COP, CVX, EOG, OXY, HP",0.872,0.170,0.263,0.377,0.944,0.112,0.158,0.303
Financial Services,586,586,28,0.184,JPM,"JPM, C, GS, WFC, USB",0.888,0.178,0.357,0.682,0.961,0.102,0.174,0.347
Healthcare,324,323,13,0.204,AMGN,"AMGN, PFE, BIIB, GILD, LLY",0.599,0.227,0.317,0.475,0.766,0.156,0.247,0.320
Industrials,270,270,2,0.956,HON,"HON, ITW, EMR, ETN, DOV",0.795,0.167,0.237,0.389,0.881,0.126,0.196,0.308
Real Estate,134,134,2,0.619,SPG,"SPG, VNO, FRT, REG, PLD",0.887,0.186,0.321,0.776,0.950,0.113,0.221,0.328
